In [ ]:
"""
================================================================================
MIXPANEL USER PROFILE EXPORTER
================================================================================

A simple tool to export all user profiles from Mixpanel to CSV via the
Engage API.

HOW TO USE:
-----------
1. Run this cell in a Jupyter notebook.
2. Enter your Mixpanel credentials when prompted:
   - Project ID (found in Mixpanel → Settings → Project Settings)
   - Service Account Username & Secret (create one in Mixpanel → Settings →
     Service Accounts → Add Service Account)
3. Optionally filter by a segmentation expression (e.g.
   'properties["$city"] == "London"') or press Enter to export all profiles.
4. Optionally specify which properties to export, or press Enter for all.
5. The tool paginates through all profiles automatically and saves a CSV
   in your notebook's working directory.

CAVEATS:
--------
- The Mixpanel Engage API is only available on the Growth plan or above.
  It will not work on the Free plan.

- The API returns at most 1000 profiles per page. This script handles
  pagination automatically using session_id, but very large profile sets
  (hundreds of thousands+) may take a while.

- Profile properties in Mixpanel can vary across users. The exported CSV
  includes a column for every property seen across all profiles. Missing
  values are left blank.

================================================================================
"""

import base64
import json
import csv
import requests
from getpass import getpass
from datetime import datetime

# ── User Inputs ──────────────────────────────────────────────
project_id = input("Mixpanel Project ID: ").strip()
username = input("Service Account Username: ").strip()
secret = getpass("Service Account Secret: ").strip()

where_filter = input(
    'Segmentation expression to filter profiles (or Enter for all): '
).strip()

props_input = input(
    "Properties to export, comma-separated (or Enter for all): "
).strip()
output_properties = None
if props_input:
    output_properties = [p.strip() for p in props_input.split(",") if p.strip()]

# ── Config ───────────────────────────────────────────────────
ENGAGE_URL = f"https://mixpanel.com/api/query/engage"
PAGE_SIZE = 1000
auth = base64.b64encode(f"{username}:{secret}".encode()).decode()
headers = {
    "Authorization": f"Basic {auth}",
    "accept": "application/json",
    "content-type": "application/x-www-form-urlencoded",
}

# ── Flatten nested properties ────────────────────────────────
def flatten(obj, prefix=""):
    out = {}
    for k, v in (obj or {}).items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            out.update(flatten(v, key))
        elif isinstance(v, list):
            out[key] = json.dumps(v)
        else:
            out[key] = v
    return out

# ── Paginate through Engage API ──────────────────────────────
print(f"\nFetching user profiles (page size: {PAGE_SIZE})...\n")

all_profiles = []
all_keys = set()
session_id = None
page = 0

while True:
    body = {
        "project_id": project_id,
        "page_size": PAGE_SIZE,
    }
    if where_filter:
        body["where"] = where_filter
    if output_properties:
        body["output_properties"] = json.dumps(output_properties)
    if session_id:
        body["session_id"] = session_id
        body["page"] = page

    r = requests.post(
        ENGAGE_URL,
        headers=headers,
        data=body,
        timeout=120,
    )

    if r.status_code != 200:
        raise RuntimeError(f"Engage API error {r.status_code}: {r.text[:300]}")

    data = r.json()

    results = data.get("results", [])
    total = data.get("total", "?")
    session_id = data.get("session_id")

    for profile in results:
        distinct_id = profile.get("$distinct_id", "")
        props = flatten(profile.get("$properties", {}))
        row = {"$distinct_id": distinct_id, **props}
        all_keys.update(row.keys())
        all_profiles.append(row)

    print(f"  Page {page + 1}: fetched {len(results)} profiles "
          f"({len(all_profiles):,} / {total} total)")

    if len(results) < PAGE_SIZE:
        break
    page += 1

print(f"\nTotal profiles fetched: {len(all_profiles):,}")
print(f"Total unique properties: {len(all_keys)}")

# ── Write CSV ────────────────────────────────────────────────
cols = ["$distinct_id"] + sorted(k for k in all_keys if k != "$distinct_id")
today = datetime.now().strftime("%Y-%m-%d")
filename = f"mixpanel_profiles_{today}.csv"

with open(filename, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=cols, extrasaction="ignore")
    writer.writeheader()
    for row in all_profiles:
        writer.writerow(row)

print(f"\nSaved to {filename}")